In [1]:
import os
import sys
import shutil
import subprocess
from datetime import datetime as dt

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

In [2]:
utils_path = !pwd
utils_path = utils_path[0]
utils_path = os.path.join(utils_path, '..', 'src', 'utils')
sys.path.append(utils_path)
utils_path

'/Users/luisenrique/Documents/network-inventory-cleaning-and-validation/notebooks/../src/utils'

In [3]:
from InventoryTransformations import InventoryTransformations

In [4]:
# # TODO: RUN THIS IF YOU HAVE MULTIPLE OPENJDK VERSIONS
# os.environ["PATH"] = f"{os.environ['JAVA_HOME']}/BIN:{os.environ.get('PATH', '')}"
# os.environ.get("JAVA_HOME")
# shutil.which("java")
# subprocess.run(["java", "-version"], capture_output=True, text=True).stderr

In [6]:
DRIVER_HOST = "host.docker.internal"
DRIVER_PORT = "4042"
BLOCK_MANAGER_PORT = "4043"

In [7]:
# TODO: MAKE SURE TO RUN THIS IN ORDER TO HANDLE host.docker.internal in local host:
#  echo "127.0.0.1 host.docker.internal" | sudo tee -a /etc/hosts 

# AWS_BUNDLE = "com.amazonaws:aws-java-sdk-bundle:1.12.262"
AWS_BUNDLE = "software.amazon.awssdk:bundle:2.23.19"
HADOOP_AWS = "org.apache.hadoop:hadoop-aws:3.4.0"

spark = (
    SparkSession.builder
    .appName("data_cleansing")
    .master("spark://localhost:7077")          # driver the notebook
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.driver.host", DRIVER_HOST)
    .config("spark.driver.port", DRIVER_PORT)
    .config("spark.blockManager.port", BLOCK_MANAGER_PORT)
    # Including dependencies
    .config("spark.submit.pyFiles", f"{utils_path}/InventoryTransformations.py")
    # ↓ Let Spark fetch & ship jars to executors
    .config("spark.jars.packages", f"{HADOOP_AWS},{AWS_BUNDLE}")
    # .config("spark.jars.packages", f"{HADOOP_AWS}")
    # MINIO/S3A
    .config("spark.hadoop.fs.s3a.endpoint", "http://host.docker.internal:9000")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    # Explicit creds provider (so the access/secret keys below are actually used)
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
    # Failure settings
    .config("spark.hadoop.fs.s3a.connection.timeout", "10000")
    .config("spark.hadoop.fs.s3a.connection.establish.timeout", "5000")
    .config("spark.hadoop.fs.s3a.socket.timeout", "30000")
    .config("spark.hadoop.fs.s3a.attempts.maximum", "3")
    .config("spark.hadoop.fs.s3a.retry.limit", "2")
    .getOrCreate()
)

spark.version, spark.sparkContext._jvm.org.apache.hadoop.util.VersionInfo.getVersion()

:: loading settings :: url = jar:file:/Users/luisenrique/Documents/network-inventory-cleaning-and-validation/venv10/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/luisenrique/.ivy2.5.2/cache
The jars for the packages stored in: /Users/luisenrique/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
software.amazon.awssdk#bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6bf73aef-5561-4a03-972b-24a36dfaa184;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.4.0 in central
	found software.amazon.awssdk#bundle;2.23.19 in central
	found org.wildfly.openssl#wildfly-openssl;1.1.3.Final in central
:: resolution report :: resolve 116ms :: artifacts dl 3ms
	:: modules in use:
	org.apache.hadoop#hadoop-aws;3.4.0 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.1.3.Final from central in [default]
	software.amazon.awssdk#bundle;2

('4.0.0', '3.4.1')

In [8]:
hadoop_config = spark._jsc.hadoopConfiguration()
print(hadoop_config.get("fs.s3a.endpoint"))
print(hadoop_config.get("fs.s3a.aws.credentials.provider"))
print(hadoop_config.get("fs.s3a.path.style.access"))
print(hadoop_config.get("fs.s3a.connection.ssl.enabled"))

http://host.docker.internal:9000
org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider
true
false


In [9]:
jvm = spark.sparkContext._jvm
fs = jvm.org.apache.hadoop.fs.FileSystem.get(jvm.java.net.URI("s3a://inventory/"), hadoop_config)
print(fs)
for s in fs.listStatus(jvm.org.apache.hadoop.fs.Path("s3a://inventory/")):
    print("->", s.getPath().toString())

25/11/13 15:48:11 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
25/11/13 15:48:11 WARN ConfigurationHelper: Option fs.s3a.connection.establish.timeout is too low (5,000 ms). Setting to 15,000 ms instead
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


S3AFileSystem{uri=s3a://inventory, workingDir=s3a://inventory/user/luisenrique, partSize=67108864, enableMultiObjectsDelete=true, maxKeys=5000, OpenFileSupport{changePolicy=ETagChangeDetectionPolicy mode=Server, defaultReadAhead=65536, defaultBufferSize=65536, defaultAsyncDrainThreshold=16000, defaultInputPolicy=default}, blockSize=33554432, multiPartThreshold=134217728, s3EncryptionAlgorithm='NONE', blockFactory=org.apache.hadoop.fs.s3a.S3ADataBlocks$DiskBlockFactory@4a3f95af, auditManager=Service ActiveAuditManagerS3A in state ActiveAuditManagerS3A: STARTED, auditor=LoggingAuditor{ID='211c0073-0450-4ad4-90c8-1edc1e6a7e59', headerEnabled=true, rejectOutOfSpan=false, isMultipartUploadEnabled=true}}, authoritativePath=[], useListV1=false, magicCommitter=true, boundedExecutor=BlockingThreadPoolExecutorService{SemaphoredDelegatingExecutor{permitCount=224, available=224, waiting=0}, activeCount=0}, unboundedExecutor=java.util.concurrent.ThreadPoolExecutor@12073369[Running, pool size = 0, a

In [10]:
# Driver JVM classpath (you'll see site-packages/pyspark/jars but no hadoop-aws/aws-sdk)
spark.sparkContext._jvm.org.apache.hadoop.util.VersionInfo.getVersion(), spark.sparkContext._jvm.java.lang.System.getProperty("java.class.path")

('3.4.1',
 'hive-jackson/*:/Users/luisenrique/Documents/network-inventory-cleaning-and-validation/venv10/lib/python3.10/site-packages/pyspark/conf:/Users/luisenrique/Documents/network-inventory-cleaning-and-validation/venv10/lib/python3.10/site-packages/pyspark/jars/slf4j-api-2.0.16.jar:/Users/luisenrique/Documents/network-inventory-cleaning-and-validation/venv10/lib/python3.10/site-packages/pyspark/jars/dropwizard-metrics-hadoop-metrics2-reporter-0.1.2.jar:/Users/luisenrique/Documents/network-inventory-cleaning-and-validation/venv10/lib/python3.10/site-packages/pyspark/jars/jjwt-api-0.12.6.jar:/Users/luisenrique/Documents/network-inventory-cleaning-and-validation/venv10/lib/python3.10/site-packages/pyspark/jars/py4j-0.10.9.9.jar:/Users/luisenrique/Documents/network-inventory-cleaning-and-validation/venv10/lib/python3.10/site-packages/pyspark/jars/metrics-jmx-4.2.30.jar:/Users/luisenrique/Documents/network-inventory-cleaning-and-validation/venv10/lib/python3.10/site-packages/pyspark/ja

In [11]:
df_raw = spark.read.csv("s3a://inventory/inventory_raw.csv", header=True, inferSchema=True)

In [12]:
df_raw.show()

+-------------+---------------+----------+-------------+-----------------+--------------------+-----------+-------------+--------------------+
|source_row_id|             ip|  hostname|         fqdn|              mac|               owner|device_type|         site|               notes|
+-------------+---------------+----------+-------------+-----------------+--------------------+-----------+-------------+--------------------+
|            1|192.168.010.005|    HOST01|         NULL|AA-BB-CC-DD-EE-FF|priya (platform) ...|     server|   BLR Campus|             db host|
|            2|     10.0.1.300|   host-02|host-02.local|11-22-33-44-55-66|                 ops|       NULL|    HQ Bldg 1|            edge gw?|
|            3|         10.0.1|    host03|         NULL|   aabb.ccdd.eeff|jane@corp.example...|     switch|HQ-BUILDING-1|                NULL|
|            4|     10.0.1.1.2|printer-01|         NULL|00:11:22:33:44:55|          Facilities|    printer|           HQ|                NULL|

### SETTING Inventory_Transformations' FUNCTIONS RETURN STRUCTS

In [13]:
traceability_schema = T.StructType([
    T.StructField("field", T.StringType(), True),
    T.StructField("from", T.StringType(), True),
    T.StructField("to", T.StringType(), True),
    T.StructField("reason", T.StringType(), True)
])

In [14]:
ipv4_schema = T.StructType([
    T.StructField("ip_valid", T.BooleanType(), True),
    T.StructField("ip_canonical", T.StringType(), True),
    T.StructField("tr_metadata", traceability_schema, True)
])

In [15]:
hostname_schema = T.StructType([
    T.StructField("hostname_valid", T.BooleanType(), True), 
    T.StructField("hostname_canonical", T.StringType(), True),
    T.StructField("tr_metadata", traceability_schema, True)
])

In [16]:
site_schema = T.StructType([
    T.StructField("site_normalized", T.StringType(), True),
    T.StructField("tr_metadata", traceability_schema, True)
])

In [17]:
fqdn_schema = T.StructType([
    T.StructField("fqdn_valid", T.BooleanType(), True),
    T.StructField("fqdn_canonical", T.StringType(), True),
    T.StructField("tr_metadata", traceability_schema, True),
    T.StructField("fqdn_consistent", T.StringType(), True)
])

In [18]:
mac_schema = T.StructType([
    T.StructField("mac_valid", T.BooleanType(), True),
    T.StructField("mac_canonical", T.StringType(), True),
    T.StructField("tr_metadata", traceability_schema, True)
])

In [19]:
owner_schema = T.StructType([
    T.StructField("owner", T.StringType(), True),
    T.StructField("owner_email", T.StringType(), True),
    T.StructField("owner_team", T.StringType(), True),
    T.StructField("tr_metadata", traceability_schema, True)
])

In [20]:
device_type_schema = T.StructType([
    T.StructField("device_type", T.StringType(), True),
    T.StructField("device_type_confidence", T.IntegerType(), True),
    T.StructField("tr_metadata", traceability_schema, True),
])

### UDFs

In [21]:
it = InventoryTransformations()

IP

In [22]:
ipv4_udf = F.udf(lambda x: it.ipv4_validate_and_normalize(x), ipv4_schema)

In [23]:
ipv4_type_udf = F.udf(lambda x: it.classify_ipv4_type(x), T.StringType())

In [24]:
default_subnet_udf = F.udf(lambda ip, ip_type: it.default_subnet(ip, ip_type), T.StringType())

HOSTNAME

In [25]:
hostname_udf = F.udf(lambda x: it.validate_hostname(x), hostname_schema)

SITE

In [26]:
site_udf = F.udf(lambda x: it.normalize_site(x), site_schema)

In [27]:
reverse_ptr_udf = F.udf(lambda x: it.generate_reverse_ptr(x), T.StringType())

FQDN

In [28]:
fqdn_udf = F.udf(lambda fqdn, hostname, site: it.validate_fqdn(fqdn, hostname, site), fqdn_schema)

MAC

In [29]:
mac_udf = F.udf(lambda x: it.validate_mac(x), mac_schema)

OWNER

In [30]:
owner_udf = F.udf(lambda x: it.parse_owner(x), owner_schema)

DEVICE_TYPE

In [31]:
device_type_udf = F.udf(lambda x: it.normalize_device_type(x), device_type_schema)

### TRANSFORMING INVENTORY

In [32]:
df = df_raw

In [33]:
df.show()

+-------------+---------------+----------+-------------+-----------------+--------------------+-----------+-------------+--------------------+
|source_row_id|             ip|  hostname|         fqdn|              mac|               owner|device_type|         site|               notes|
+-------------+---------------+----------+-------------+-----------------+--------------------+-----------+-------------+--------------------+
|            1|192.168.010.005|    HOST01|         NULL|AA-BB-CC-DD-EE-FF|priya (platform) ...|     server|   BLR Campus|             db host|
|            2|     10.0.1.300|   host-02|host-02.local|11-22-33-44-55-66|                 ops|       NULL|    HQ Bldg 1|            edge gw?|
|            3|         10.0.1|    host03|         NULL|   aabb.ccdd.eeff|jane@corp.example...|     switch|HQ-BUILDING-1|                NULL|
|            4|     10.0.1.1.2|printer-01|         NULL|00:11:22:33:44:55|          Facilities|    printer|           HQ|                NULL|

IPV4 TRANSFORMATIONS

In [34]:
df = (df.withColumn("ipv4", ipv4_udf(F.col("ip")))
      .withColumn("ip_valid", F.col("ipv4.ip_valid"))
      .withColumn("ip_canonical", F.col("ipv4.ip_canonical"))
      .withColumn("ip_tr_metadata", F.col("ipv4.tr_metadata"))
      .drop('ipv4'))

In [35]:
df = df.withColumn("ip_type", ipv4_type_udf(F.col("ip_canonical")))

In [36]:
df = df.withColumn("subnet_cidr", F.when(F.col("ip_valid") == True, default_subnet_udf(F.col('ip_canonical'), F.col("ip_type"))).otherwise(F.lit(None)))


HOSTNAME TRANSFORMATIONS

In [37]:
df = (
    df.withColumn("hr", hostname_udf(F.col("hostname")))
        .withColumn("hostname_valid", F.col("hr.hostname_valid"))
        .withColumn("hostname_canonical", F.col("hr.hostname_canonical"))
        .withColumn("hostname_tr_metadata", F.col("hr.tr_metadata"))
        .drop("hr")
)

SITE TRANSFORMATIONS

In [38]:
df = (
    df.withColumn("site_res", site_udf(F.col("site")))
        .withColumn("site_normalized", F.col("site_res.site_normalized"))
        .withColumn("site_tr_metadata", F.col("site_res.tr_metadata"))
        .drop("site_res")
)


FQDN TRANSFORMATIONS

In [39]:
df = df.withColumn("reverse_ptr", reverse_ptr_udf(F.col("ip_canonical")))

In [40]:

df = (
    df.withColumn("fqdn_res", fqdn_udf(F.col('fqdn'), F.col("hostname_canonical"), F.col("site_normalized")))
        .withColumn("fqdn_valid", F.col("fqdn_res.fqdn_valid"))
        .withColumn("fqdn_canonical", F.col("fqdn_res.fqdn_canonical"))
        .withColumn("fqdn_tr_metadata", F.col("fqdn_res.tr_metadata"))
        .withColumn("fqdn_consistent", F.col("fqdn_res.fqdn_consistent"))
        .drop("fqdn_res")
)

MAC TRANSFORMATIONS

In [41]:
df = (
    df.withColumn("mac_res", mac_udf(F.col("mac")))
        .withColumn("mac_valid", F.col("mac_res.mac_valid"))
        .withColumn("mac_canonical", F.col("mac_res.mac_canonical"))
        .withColumn("mac_tr_metadata", F.col("mac_res.tr_metadata"))
        .drop("mac_res")
)


OWNER PARSING

In [42]:
df = (
    df.withColumn("owner_res", owner_udf(F.col("owner")))
        .withColumn("owner_normalized", F.col("owner_res.owner"))
        .withColumn("owner_email", F.col("owner_res.owner_email"))
        .withColumn("owner_team", F.col('owner_res.owner_team'))
        .withColumn("owner_tr_metadata", F.col("owner_res.tr_metadata"))
        .drop("owner_res")
)


DEVICE TYPE TRANSFORMATIONS

In [43]:
df = (
    df.withColumn("dt_res", device_type_udf(F.col("device_type")))
        .withColumn("device_type_normalized", F.col("dt_res.device_type"))
        .withColumn("device_type_confidence", F.col("dt_res.device_type_confidence"))
        .withColumn("device_type_tr_metadata", F.col("dt_res.tr_metadata"))
        .drop("dt_res")
)


NORMALIZATION STEPS

In [44]:
df = df.withColumn(
    "normalization_steps",
    F.array(
        "ip_tr_metadata",
        "hostname_tr_metadata",
        "fqdn_tr_metadata",
        "mac_tr_metadata",
        "owner_tr_metadata",
        "device_type_tr_metadata",
        "site_tr_metadata"
    )
)

In [45]:
df.printSchema()

root
 |-- source_row_id: integer (nullable = true)
 |-- ip: string (nullable = true)
 |-- hostname: string (nullable = true)
 |-- fqdn: string (nullable = true)
 |-- mac: string (nullable = true)
 |-- owner: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- site: string (nullable = true)
 |-- notes: string (nullable = true)
 |-- ip_valid: boolean (nullable = true)
 |-- ip_canonical: string (nullable = true)
 |-- ip_tr_metadata: struct (nullable = true)
 |    |-- field: string (nullable = true)
 |    |-- from: string (nullable = true)
 |    |-- to: string (nullable = true)
 |    |-- reason: string (nullable = true)
 |-- ip_type: string (nullable = true)
 |-- subnet_cidr: string (nullable = true)
 |-- hostname_valid: boolean (nullable = true)
 |-- hostname_canonical: string (nullable = true)
 |-- hostname_tr_metadata: struct (nullable = true)
 |    |-- field: string (nullable = true)
 |    |-- from: string (nullable = true)
 |    |-- to: string (nullable = true)
 |

In [46]:
df = df.withColumn("normalization_steps", F.to_json(F.col("normalization_steps")))
df.select("normalization_steps").show(truncate=False)

+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|normalization_steps                                                                                                                                                                                                                                                                                                                                                                                                                                                       

DF_FINAL

In [47]:
df_final = df.select(
    F.col("ip_canonical").alias("ip"),
    F.col("ip_valid"),
    F.col("ip_type"),
    F.col("subnet_cidr"),
    F.col("hostname_canonical").alias("hostname"),
    F.col("hostname_valid"),
    F.col("fqdn_canonical").alias("fqdn"),
    F.col("fqdn_consistent"),
    F.col("reverse_ptr"),
    F.col("mac_canonical").alias("mac"),
    F.col("mac_valid"),
    F.col("owner_normalized").alias("owner"),
    F.col("owner_email"),
    F.col("owner_team"),
    F.col("device_type_normalized").alias("device_type"),
    F.col("device_type_confidence"),
    F.col("site"),
    F.col("site_normalized"),
    F.col("source_row_id"),
    F.col("normalization_steps"),
)
df_final.show()

+-------------+--------+----------------+---------------+----------+--------------+-------------+---------------+--------------------+-----------------+---------+----------+--------------------+----------+-----------+----------------------+-------------+---------------+-------------+--------------------+
|           ip|ip_valid|         ip_type|    subnet_cidr|  hostname|hostname_valid|         fqdn|fqdn_consistent|         reverse_ptr|              mac|mac_valid|     owner|         owner_email|owner_team|device_type|device_type_confidence|         site|site_normalized|source_row_id| normalization_steps|
+-------------+--------+----------------+---------------+----------+--------------+-------------+---------------+--------------------+-----------------+---------+----------+--------------------+----------+-----------+----------------------+-------------+---------------+-------------+--------------------+
| 192.168.10.5|    true| private_rfc1918|192.168.10.0/24|    host01|          true

### EXPORTING CSV

In [48]:
curr_path = !pwd
curr_path = curr_path[0]
out_path = os.path.join(curr_path, f"{dt.today().year}-{dt.today().day}-{dt.today().month}_inventory_tmp.csv")
# (df_final
#  .coalesce(1)
#  .write
#  .option("header", True)
#  .mode("overwrite")
#  .csv(out_path))

### COMPARING RESULTS WITH PANDAS TRANSFORMATIONS

In [49]:
# input was manually uploaded to minio
pandas_df = spark.read.csv("s3a://inventory/01-ingest-and-transform-inventory/pandas_df.csv", header=True, inferSchema=True)
pandas_df = pandas_df.drop("_c0")
pandas_df.show(truncate=False)

+-------------+--------+----------------+---------------+----------+--------------+-------------+---------------+--------------------------+-----------------+---------+----------+----------------------+----------+-----------+----------------------+-------------+---------------+-------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|ip           |ip_valid|ip_type         |subnet_cidr    |hostname  |hostname_valid|fqdn     

In [50]:
def compare_dataframes(df1, df2, key_cols="source_row_id", float_tol=None):
    # 1) Align rows
    l, r = df1.alias("l"), df2.alias("r")
    j = l.join(r, on=key_cols, how="inner")

    # 2) Build equality columns (null-safe). Optional tolerance for floats.
    common_cols = [c for c in df1.columns if c in df2.columns and c not in key_cols]
    eq_cols = []
    for c in common_cols:
        lc, rc = F.col(f"l.{c}"), F.col(f"r.{c}")
        if float_tol is not None and dict(j.dtypes)[f"l.{c}"] in ("double","float") and dict(j.dtypes)[f"r.{c}"] in ("double","float"):
            # null-safe approx equality: both null OR |l-r|<=tol
            eq = (lc.isNull() & rc.isNull()) | (lc.isNotNull() & rc.isNotNull() & (F.abs(lc - rc) <= F.lit(float_tol)))
        else:
            eq = lc.eqNullSafe(rc)
        eq_cols.append(eq.cast("int").alias(f"eq_{c}"))

    compared = j.select(*key_cols, *eq_cols)

    # 3) Per-column match counts and overall similarity
    per_col = compared.agg(*[F.sum(f"eq_{c}").alias(c) for c in common_cols])
    n_rows = compared.count()
    n_cols = len(common_cols)

    # overall equals = sum of all per-column equals
    sums = per_col.collect()[0].asDict()
    total_equal = sum(sums.values())
    similarity = total_equal / float(n_rows * n_cols) if n_rows and n_cols else 0.0

    return {
        "rows_compared": n_rows,
        "columns_compared": n_cols,
        "overall_equal_cells": total_equal,
        "overall_similarity": similarity,  # 0..1
        "per_column_equal_counts": sums,
        "per_column_match_rate": {c: sums[c] / float(n_rows) if n_rows else 0.0 for c in common_cols},
        "details_df": compared,  # optional: one-hot match flags per row/column
    }

In [51]:
compare_dataframes(df_final, pandas_df, ["source_row_id"])

{'rows_compared': 15,
 'columns_compared': 19,
 'overall_equal_cells': 269,
 'overall_similarity': 0.9438596491228071,
 'per_column_equal_counts': {'ip': 15,
  'ip_valid': 15,
  'ip_type': 15,
  'subnet_cidr': 15,
  'hostname': 15,
  'hostname_valid': 15,
  'fqdn': 15,
  'fqdn_consistent': 15,
  'reverse_ptr': 15,
  'mac': 15,
  'mac_valid': 15,
  'owner': 15,
  'owner_email': 15,
  'owner_team': 15,
  'device_type': 15,
  'device_type_confidence': 15,
  'site': 14,
  'site_normalized': 15,
  'normalization_steps': 0},
 'per_column_match_rate': {'ip': 1.0,
  'ip_valid': 1.0,
  'ip_type': 1.0,
  'subnet_cidr': 1.0,
  'hostname': 1.0,
  'hostname_valid': 1.0,
  'fqdn': 1.0,
  'fqdn_consistent': 1.0,
  'reverse_ptr': 1.0,
  'mac': 1.0,
  'mac_valid': 1.0,
  'owner': 1.0,
  'owner_email': 1.0,
  'owner_team': 1.0,
  'device_type': 1.0,
  'device_type_confidence': 1.0,
  'site': 0.9333333333333333,
  'site_normalized': 1.0,
  'normalization_steps': 0.0},
 'details_df': DataFrame[source_row_

- There's a discrepancy in how the Spark DataFrame loads the "site" column compared to how the pandas library does it. Thus, we observe the 'site' : 14/15
- The 'normalization_steps' column keeps the information, a slight discrepancy in string format is making the per_column_match_rate: 0.0